# Sports-only resampled booster with the 15-neighbor specialist
Three fixed rules compare an equal vote for the resampled booster with the existing specialist. Conditional blending preserves the total sports probability. Direct blending averages the two sports scores independently before comparing to the unchanged other-class scores. The gated rule permits only switches between previously predicted sports.

In [1]:
from pathlib import Path
from collections import deque
import zipfile
import shutil
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 610
import os
WORKERS = min(8, os.cpu_count() or 1)
print("scikit-learn", sklearn.__version__)
def locate(name):
    for root in (Path.cwd(), Path.cwd().parent):
        direct = root / name
        if direct.is_file():
            return direct
        data = root / "Docs and Data"
        if data.is_dir():
            match = next(data.rglob(name), None)
            if match:
                return match

har_path=locate("har_train.csv")
if not har_path:
    from google.colab import files
    files.upload()
    har_path=locate("har_train.csv")
har=pd.read_csv(har_path,header=None)


scikit-learn 1.9.1


In [2]:
import importlib.util
import subprocess
import sys
if any(importlib.util.find_spec(k) is None for k in ("imblearn","flaml","xgboost","lightgbm")):
    subprocess.check_call([sys.executable,"-m","pip","install","imbalanced-learn==0.14.2",
                           "flaml==2.8.0","xgboost","lightgbm"])
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from flaml.automl.model import XGBoostSklearnEstimator, LGBMEstimator
from collections import Counter

def booster(family, config, task):
    wrapper = {"xgboost":XGBoostSklearnEstimator, "lgbm":LGBMEstimator}[family](
        task=task,n_jobs=WORKERS,**config)
    return wrapper.estimator_class(**wrapper.params)

def resample(matrix, y, strategy, k):
    sampler = SMOTETomek(smote=SMOTE(sampling_strategy=strategy,k_neighbors=k,random_state=SEED),
                         tomek=TomekLinks(sampling_strategy="all",n_jobs=1),random_state=SEED)
    return sampler.fit_resample(matrix,y)

activity_configs=[('xgboost', {'n_estimators': 56, 'max_leaves': 4, 'min_child_weight': 2.707170950755067, 'learning_rate': 0.4810290992611119, 'subsample': 0.9659488304727971, 'colsample_bylevel': 1.0, 'colsample_bytree': 0.9692046632582698, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.1384884162470523}), ('lgbm', {'n_estimators': 100, 'num_leaves': 4, 'min_child_samples': 16, 'learning_rate': 0.07832503615697724, 'log_max_bin': 9, 'colsample_bytree': 0.8814460015050225, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.024037654653852953}), ('xgboost', {'n_estimators': 56, 'max_leaves': 4, 'min_child_weight': 2.707170950755067, 'learning_rate': 0.4810290992611119, 'subsample': 0.9659488304727971, 'colsample_bylevel': 1.0, 'colsample_bytree': 0.9692046632582698, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.1384884162470523}), ('xgboost', {'n_estimators': 56, 'max_leaves': 4, 'min_child_weight': 2.707170950755067, 'learning_rate': 0.4810290992611119, 'subsample': 0.9659488304727971, 'colsample_bylevel': 1.0, 'colsample_bytree': 0.9692046632582698, 'reg_alpha': 0.0009765625, 'reg_lambda': 0.1384884162470523})]


In [3]:
Xh, yh = har.iloc[:,:-1], har.iloc[:,-1].to_numpy()
groups = pd.util.hash_pandas_object(Xh, index=False).to_numpy()
labels = np.unique(yh)
# Accept a split only if all activities occur on both sides of every fold.
for split_seed in range(SEED, SEED+200):
    splitter = StratifiedGroupKFold(4, shuffle=True, random_state=split_seed)
    har_folds = list(splitter.split(Xh, yh, groups))
    if all(len(np.unique(yh[tr])) == len(labels) == len(np.unique(yh[va])) for tr,va in har_folds):
        break
else:
    raise ValueError("No four-fold split covers all activities.")
print("Fold seed:", split_seed)
labels = np.unique(yh)
ycode = np.searchsorted(labels,yh)


Fold seed: 610


In [4]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import NearestNeighbors
from sklearn.utils.class_weight import compute_sample_weight
def row_summaries(z):
    return np.column_stack([np.mean(abs(z),axis=1),np.sqrt(np.mean(z*z,axis=1)),
                            np.mean(abs(z)>2,axis=1),np.quantile(z,[.1,.5,.9],axis=1).T])

def shares(ref,query,ids,group_ids,project,k=15):
    scale=StandardScaler().fit(ref)
    keep=scale.var_>0
    a=scale.transform(ref)[:,keep]
    b=scale.transform(query)[:,keep]
    if project:
        lda=LinearDiscriminantAnalysis(solver="eigen",shrinkage="auto").fit(a,ids)
        a,b=lda.transform(a),lda.transform(b)
    # Equal contribution per distinct reading, even if it appears many times.
    unique,first,inv=np.unique(group_ids,return_index=True,return_inverse=True)
    votes=np.zeros((len(unique),len(labels)))
    np.add.at(votes,(inv,ids),1)
    votes/=votes.sum(axis=1,keepdims=True)
    distance,near=NearestNeighbors(n_neighbors=k).fit(a[first]).kneighbors(b)
    weight=1/np.maximum(distance,1e-8)
    return np.sum(votes[near]*weight[:,:,None],axis=1)/weight.sum(axis=1,keepdims=True)

# Fixed candidates: no outer labels are used to set blend weights or choose features.
baseline="15 shares"
joint="10 + 15 shares"
radius="15 shares + distances"
blend="15-share booster + resampled booster"
specialist="15 shares + Tennis/Football specialist"
names=[baseline,joint,radius,blend,specialist]
preds={name:np.empty_like(ycode) for name in names}
extra=["Conditional sports blend","Direct sports score blend","Sports predictions only"]
preds.update({name:np.empty_like(ycode) for name in extra})
fold_ids=np.zeros(len(ycode),int)
pair=np.array([np.flatnonzero(labels==name)[0] for name in ("Football","Tennis")])

def distance_features(ref,query,groups_ref):
    scale=StandardScaler().fit(ref)
    keep=scale.var_>0
    _,first=np.unique(groups_ref,return_index=True)
    a=scale.transform(ref[first])[:,keep]
    b=scale.transform(query)[:,keep]
    d=NearestNeighbors(n_neighbors=15).fit(a).kneighbors(b)[0]
    return np.column_stack([d[:,0],d[:,-1],d[:,0]/np.maximum(d[:,-1],1e-8)])

def neighborhood(ref,query,y,groups_ref):
    return np.column_stack([shares(ref,query,y,groups_ref,False,k) for k in (10,15)]+
                           [distance_features(ref,query,groups_ref)])

for fold,(tr,va) in enumerate(har_folds):
    assert not np.intersect1d(groups[tr],groups[va]).size
    fold_ids[va]=fold+1
    x,v=Xh.iloc[tr].to_numpy(),Xh.iloc[va].to_numpy()
    y=ycode[tr];w=compute_sample_weight("balanced",y)
    scale=StandardScaler().fit(x);xs,vs=scale.transform(x),scale.transform(v)
    for seed in range(SEED,SEED+200):
        inner=list(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(x,y,groups[tr]))
        if all(len(np.unique(y[a]))==len(labels) for a,b in inner):break
    else:raise ValueError("Missing reference class")
    m=len(labels)
    meta=np.empty((len(tr),2*m+3))
    for a,b in inner:
        assert not np.intersect1d(groups[tr[a]],groups[tr[b]]).size
        meta[b]=neighborhood(x[a],x[b],y[a],groups[tr[a]])
    vm=neighborhood(x,v,y,groups[tr])
    fam,cfg=activity_configs[fold]
    for name,ix in [(baseline,slice(m,2*m)),(joint,slice(0,2*m)),(radius,slice(m,None))]:
        scale_meta=StandardScaler().fit(meta[:,ix].copy())
        xx=np.column_stack((xs,scale_meta.transform(meta[:,ix].copy())))
        vv=np.column_stack((vs,scale_meta.transform(vm[:,ix].copy())))
        model=booster(fam,cfg,"multiclass").fit(xx,y,sample_weight=w)
        prob=model.predict_proba(vv)
        preds[name][va]=model.classes_[prob.argmax(axis=1)]
        if name==baseline:
            base_prob=prob.copy()
            # Conditional classifier changes only the division of probability between the two sports.
            mask=np.isin(y,pair)
            sports=booster(fam,cfg,"binary").fit(xx[mask],(y[mask]==pair[1]).astype(int))
            updated=base_prob.copy()
            mass=base_prob[:,pair].sum(axis=1,keepdims=True)
            updated[:,pair]=0.5*(base_prob[:,pair]+mass*sports.predict_proba(vv))
            preds[specialist][va]=updated.argmax(axis=1)
    counts=Counter(y); target=int(np.median(list(counts.values())))
    strategy={c:target for c,n in counts.items() if n<target}
    xr,yr=resample(xs,y,strategy,min(3,min(counts.values())-1))
    resampled=booster(fam,cfg,"multiclass").fit(xr,yr)
    preds[blend][va]=(base_prob+resampled.predict_proba(vs)).argmax(axis=1)
    r=resampled.predict_proba(vs)[:,pair]
    scores=updated.copy()
    scores[:,pair]=0.5*(updated[:,pair]+mass*r/np.maximum(r.sum(axis=1,keepdims=True),1e-15))
    preds[extra[0]][va]=scores.argmax(axis=1)
    scores[:,pair]=0.5*(updated[:,pair]+r)
    preds[extra[1]][va]=scores.argmax(axis=1)
    previous=preds[specialist][va]
    preds[extra[2]][va]=np.where(np.isin(previous,pair),pair[scores[:,pair].argmax(axis=1)],previous)
base_ok=preds[specialist]==ycode
print("Reference correct and macro F1:",base_ok.sum(),f1_score(ycode,preds[specialist],average="macro"))
results=[];pairs=[]
for name in [specialist]+extra:
    p=preds[name]
    ok=p==ycode
    results.append(dict(model=name,correct=int(ok.sum()),accuracy=ok.mean(),macro_f1=f1_score(ycode,p,average="macro"),
                        fold_net=[int(ok[fold_ids==f].sum()-base_ok[fold_ids==f].sum()) for f in range(1,5)]))
    for activity in ("Tennis","Football"):
        mask=yh==activity
        pairs.append(dict(model=name,activity=activity,correct=int((mask&ok).sum()),
                          fixed=int((mask&ok&~base_ok).sum()),introduced=int((mask&~ok&base_ok).sum())))
print(pd.DataFrame(results).to_string(index=False))
print(pd.DataFrame(pairs).to_string(index=False))

Reference correct and macro F1: 1965 0.7092623124400687
                                 model  correct  accuracy  macro_f1       fold_net
15 shares + Tennis/Football specialist     1965  0.804668  0.709262   [0, 0, 0, 0]
              Conditional sports blend     1966  0.805078  0.709189 [2, 2, -1, -2]
             Direct sports score blend     1976  0.809173  0.713952 [10, 2, 1, -2]
               Sports predictions only     1966  0.805078  0.709497  [3, 1, 0, -3]
                                 model activity  correct  fixed  introduced
15 shares + Tennis/Football specialist   Tennis      428      0           0
15 shares + Tennis/Football specialist Football      316      0           0
              Conditional sports blend   Tennis      428     10          10
              Conditional sports blend Football      318      9           7
             Direct sports score blend   Tennis      434     16          10
             Direct sports score blend Football      320     11          